# GDA Entity Exploration: Locations & Sites

Welcome! This notebook walks through:
1. Establishing a connection to the development SQLite database using `DatabaseSessionManager`.
2. Loading `Locations` and `Sites` collections directly from live database tables.
3. Exploring lookup, filtering, querying, and lifecycle capabilities on live collections.

Code cells are kept small, focused, and bite-sized for educational purposes.

## 1. Environment & Path Setup

First, we add `src` to Python's search path (`sys.path`) so we can import our workspace modules cleanly.

In [7]:
import sys
from pathlib import Path

# Locate the repository root and add src/ to the path
REPO_ROOT = Path("G:/My Drive/genealogy-workspace")
SRC_PATH = REPO_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print(f"Workspace Root: {REPO_ROOT}")
print(f"Source Path:    {SRC_PATH}")

Workspace Root: G:\My Drive\genealogy-workspace
Source Path:    G:\My Drive\genealogy-workspace\src


## 2. Database Connection & Schema Initialization

`DatabaseSessionManager` manages SQLite database connections. `SchemaManager` verifies that all canonical tables (including `locations` and `sites`) exist.

In [8]:
from gda.db.schema import SchemaManager
from gda.db.session import DatabaseSessionManager

# Initialize session manager pointing to the default dev database
session = DatabaseSessionManager()
schema_mgr = SchemaManager(session_manager=session)

# Ensure tables and indices are initialized
schema_mgr.init_db()
print("Database session initialized and schema verified.")

Database session initialized and schema verified.


Let's test our live connection by executing a simple query.

In [9]:
with session.get_connection() as conn:
    cursor = conn.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;")
    tables = [row["name"] for row in cursor.fetchall()]

print("Active tables:", tables)

Active tables: ['fact_participant_xrefs', 'facts', 'families', 'individual_family_xrefs', 'individuals', 'locations', 'media_xrefs', 'schema_migrations', 'sites', 'source_records']


## 3. Loading the Locations Collection

Civil jurisdictions are modeled using `Locations`. We can load all records from the database using the factory method `Locations.from_database()`.

In [10]:
from gda.entity.locations.collection import Locations

# Load all locations from the SQLite database
locations = Locations.from_database()
print(f"Loaded {len(locations)} active location(s).")
print(f"Collection is dirty? {locations.is_dirty}")

Loaded 0 active location(s).
Collection is dirty? False


### 3.1 Inspecting Location Entities

Let's iterate through the collection to inspect a location's ID, canonical name, and jurisdiction tier.

In [11]:
for loc in list(locations)[:5]:
    print(f"[{loc.id}] {loc.name.canonical} ({loc.jurisdiction_tier})")

### 3.2 Finding Locations by Identifier or URN

`Locations` supports membership checks (`in`) and key access (`[...]`) by `LocationID`, string ID, or canonical URN.

In [12]:
sample_loc = next(iter(locations), None)

if sample_loc:
    # Check membership with string ID
    print("Found by ID?", str(sample_loc.id) in locations)

    # Check membership with canonical URN
    print("Found by URN?", sample_loc.canonical_urn in locations)

    # Direct retrieval
    retrieved = locations[sample_loc.id]
    print(f"Retrieved: {retrieved.name.canonical}")

### 3.3 Searching and Querying Locations

We can query locations by tier or find specific jurisdictions.

In [13]:
from gda.enums.location import JurisdictionTier

# Find all state-level jurisdictions
states = [loc for loc in locations if loc.jurisdiction_tier == JurisdictionTier.STATOID]
print(f"Found {len(states)} state-level jurisdiction(s):")
for s in states[:5]:
    print(f"  - {s.name.canonical}")

ImportError: cannot import name 'JurisdictionTier' from 'gda.enums.location' (G:\My Drive\genealogy-workspace\src\gda\enums\location.py)

## 4. Loading the Sites Collection

Physical structures, cemeteries, and landmarks are modeled using `Sites`. Each site links to a governing `LocationID`.

In [ ]:
from gda.entity.sites.collection import Sites

# Load all sites from the SQLite database
sites = Sites.from_database()
print(f"Loaded {len(sites)} active site(s).")
print(f"Collection is dirty? {sites.is_dirty}")

### 4.1 Inspecting Sites

Let's inspect the active sites, their categories, and their linked locations.

In [ ]:
for site in list(sites)[:5]:
    print(
        f"[{site.id}] {site.name.canonical} | Type: {site.category.value} | Loc: {site.location_id}"
    )

### 4.2 Querying Sites by Category

`Sites` maintains an internal index for categories, allowing fast lookups via `find_by_category()`.

In [ ]:
from gda.enums.site import SiteType

# Query for churches or cemeteries
churches = sites.find_by_category(SiteType.CHURCH)
print(f"Found {len(churches)} church site(s):")
for c in churches:
    print(f"  - {c.name.canonical} (ID: {c.id})")

### 4.3 Reverse Querying Sites by Governing Location

We can easily find all physical sites situated within a specific civil jurisdiction using `find_by_location()`.

In [ ]:
if sample_loc:
    local_sites = sites.find_by_location(sample_loc.id)
    print(f"Sites situated in '{sample_loc.name.canonical}' ({sample_loc.id}): {len(local_sites)}")
    for ls in local_sites:
        print(f"  - {ls.name.canonical} ({ls.category.value})")

## 5. Adding New Entities to a Live Collection

Let's demonstrate creating and adding a new site using the collection factory method `create()`.

`create()` automatically mints sequential identifiers (e.g., `SID-000001`), computes the canonical URN, and stages the entity in memory.

In [ ]:
from gda.model.atom.id import LocationID

# Ensure we have a valid LocationID to link against
target_loc_id = sample_loc.id if sample_loc else LocationID("LOC-000001")

new_site = sites.create(
    name="Old Bow Town Meetinghouse",
    category=SiteType.CHURCH,
    location_id=target_loc_id,
    alternate_names=["Bow Center Meetinghouse", "Old First Church"],
)

print(f"Created Site ID:     {new_site.id}")
print(f"Canonical URN:       {new_site.canonical_urn}")
print(f"Aliases:             {new_site.aliases}")
print(f"Collection is dirty? {sites.is_dirty}")

### 5.1 Alias Resolution

The collection maintains an alias index so sites can be resolved by colloquial or alternate names.

In [ ]:
# The collection indexes aliases in lowercase for case-insensitive lookup
resolved_id = sites._alias_index.get("bow center meetinghouse")
print(f"Alias 'Bow Center Meetinghouse' resolves to SiteID: {resolved_id}")

if resolved_id:
    print(f"Resolved Site Name: {sites[resolved_id].name.canonical}")

## 6. Lifecycle Management: Soft-Delete (Tombstoning) & Undelete

In the GDA architecture, entities are never deleted destructively. Instead, they are retired using in-place **tombstoning**.

Let's retire our newly created site.

In [ ]:
# Soft-delete (retire) the site
retired_site = sites.soft_delete(
    new_site.id, reason="Historical structure destroyed by fire in 1895"
)

print(f"Site is deprecated? {retired_site.tombstone.is_deprecated}")
print(f"Deprecation Reason:  {retired_site.tombstone.reason}")
print(f"Active sites count:  {len(sites)}")

Notice that `len(sites)` and standard iteration only yield active records. To inspect all records including tombstones, use `iter_all()`.

In [ ]:
all_sites = list(sites.iter_all())
tombstoned = [s for s in all_sites if s.tombstone.is_deprecated]

print(f"Total Sites (Active + Tombstoned): {len(all_sites)}")
print(f"Tombstoned Sites Count:            {len(tombstoned)}")

Now let's restore the site back to active status using `undelete()`.

In [ ]:
# Reactivate the site
restored_site = sites.undelete(new_site.id, locations_collection=locations)

print(f"Site is deprecated? {restored_site.tombstone.is_deprecated}")
print(f"Active sites count:  {len(sites)}")
print(f"Can retrieve again?  {sites[new_site.id].name.canonical}")

## 7. Interactive Discovery via `.help()`

Every core service class inherits `IntrospectableMixin`, allowing immediate terminal and notebook inspection of available methods and properties without checking private internals.

In [14]:
# Print the public directory for Sites
print(sites.help(as_str=True))

NameError: name 'sites' is not defined

In [ ]:
# Print the public directory for Locations
print(locations.help(as_str=True))